# ISL → English: pipeline runner

Use this notebook during the **future execution phase** to run the existing CLI one stage at a time. It contains no separate model/training implementation. Nothing in this notebook has been executed or numerically validated.

Every command cell defaults to `execute=False` and previews a command. Change that argument to `True` only for the stage you intend to run. Model preparation additionally requires `allow_network=True`. Saved-report cells only read files that already exist.

Start with the CE-only baseline, then select an explicit experiment configuration when needed. Follow the [runbook](../docs/runbook.md) and [data contracts](../docs/data_contracts.md).


## 1. Future environment setup

Run these commands **in a terminal**, from the repository root, after confirming compatibility of the intended AMD host:

```bash
uv lock
uv sync --locked --extra rocm --extra notebooks
uv run --no-sync --env-file .env jupyter lab notebooks/run_pipeline.ipynb
```

Create a local `.env` from `.env.example` first if it does not already exist. Fill it locally; do not paste credentials into this notebook. Use `--extra cpu` instead of `--extra rocm` for CPU. Include required experiment extras in the same sync command.

Select the project's `.venv` Python kernel. In VS Code, choose **Select Kernel → Python Environments → .venv/bin/python**. If JupyterLab does not offer it, register the interpreter from the terminal and select **ISL SLT (uv)**:

```bash
uv run --no-sync python -m ipykernel install --user --name isl-slt --display-name "ISL SLT (uv)"
```

No executable notebook cell installs dependencies or starts Jupyter. See [AMD/uv setup](../docs/amd_setup.md) and [uv's Jupyter guide](https://docs.astral.sh/uv/guides/integration/jupyter/).

Managed Colab/Kaggle users: read [managed setup](../docs/colab.md), set `RUNNER="managed"` and keep the host torch build. All installation commands remain terminal instructions.


## 2. Select the experiment

Edit `CONFIG_PATH` once; for example, choose `configs/local_hand.yaml` for wrist-local hand features. Supply exact dataset revisions, paths, export columns and verified source mappings before executing preparation. Defaults marked `REQUIRED` intentionally fail auditing.

Leave `TRAINING_SEED=None` to use the config's seed/run directory. A seed override uses the same `seed-<seed>` directory convention as the CLI. `CHECKPOINT_NAME` is relative to that run's `checkpoints/` directory.

`DECODING_FILE=None` selects this run/checkpoint's derived validation/decoding.json path. Execute final validation selection before test/gold. For averaged or otherwise reselected decoding, run the separate validation-selection cell first and set `DECODING_FILE` to the printed validation-selection path. Rerun configuration and helper cells after changing parameters. The setup cell imports only the config loader, not models or training code.


In [ ]:
from pathlib import Path
import json
import os
import shlex
import shutil
import signal
import subprocess
import sys

RUNNER = "uv"  # "uv" for local/AMD; "managed" for Colab/Kaggle
CONFIG_PATH = "configs/baseline.yaml"
TRAINING_SEED = None
ALLOW_CODE_CHANGE = False  # Resume only; never bypasses environment/data checks.
EVALUATION_PARTITION = "test"  # "validation", "test", or "gold"
CHECKPOINT_NAME = "best-bleu.pt"
DECODING_FILE = None

working_directory = Path.cwd().resolve()
REPO_ROOT = next(
    (candidate for candidate in (working_directory, *working_directory.parents)
     if (candidate / "pyproject.toml").is_file()
     and (candidate / "src" / "isl_slt" / "cli.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook from the repository root or notebooks/ directory.")
if RUNNER == "uv":
    if Path(sys.prefix).resolve() != (REPO_ROOT / ".venv").resolve():
        raise RuntimeError("Select this repository's .venv Python kernel before continuing.")
elif RUNNER == "managed":
    from isl_slt.utils.environment import check_managed_environment
    print(json.dumps(check_managed_environment(), indent=2))
else:
    raise ValueError("RUNNER must be uv or managed.")

from isl_slt.config import load_config

def repo_path(value):
    path = Path(value).expanduser()
    return (path if path.is_absolute() else REPO_ROOT / path).resolve()

CONFIG_FILE = repo_path(CONFIG_PATH)
if not CONFIG_FILE.is_file():
    raise FileNotFoundError(f"Configuration not found: {CONFIG_FILE}")
CONFIG = load_config(CONFIG_FILE)
if TRAINING_SEED is not None and (
    isinstance(TRAINING_SEED, bool) or not isinstance(TRAINING_SEED, int)
    or not 0 <= TRAINING_SEED <= 4294967295
):
    raise ValueError("TRAINING_SEED must be None or an integer in [0, 4294967295].")
if EVALUATION_PARTITION not in ("validation", "test", "gold"):
    raise ValueError("Choose validation, test, or gold for EVALUATION_PARTITION.")

RUN_DIR = repo_path(CONFIG.run_dir)
if TRAINING_SEED is not None:
    RUN_DIR = RUN_DIR / f"seed-{TRAINING_SEED}"
CHECKPOINT = RUN_DIR / "checkpoints" / CHECKPOINT_NAME
RESUME_CHECKPOINT = RUN_DIR / "checkpoints" / "last.pt"
EVALUATION_BASE = RUN_DIR / "evaluation" / CHECKPOINT.stem
EVALUATION_OUTPUT = EVALUATION_BASE / EVALUATION_PARTITION
VALIDATION_OUTPUT = EVALUATION_BASE / "validation"
DIAGNOSTIC_OUTPUT = RUN_DIR / "diagnostics" / CHECKPOINT.stem / EVALUATION_PARTITION
SPLIT_DIR = repo_path(CONFIG.data.split_dir)
ENV_FILE = REPO_ROOT / ".env"

print(json.dumps({
    "repository": str(REPO_ROOT),
    "config": str(CONFIG_FILE),
    "experiment": CONFIG.experiment,
    "seed": CONFIG.training.seed if TRAINING_SEED is None else TRAINING_SEED,
    "run_directory": str(RUN_DIR),
    "checkpoint": str(CHECKPOINT),
    "resume_checkpoint": str(RESUME_CHECKPOINT),
    "evaluation_output": str(EVALUATION_OUTPUT),
    "validation_decoding_file": str(VALIDATION_OUTPUT / "decoding.json"),
    "decoder": CONFIG.model.decoder,
    "rgb_mode": CONFIG.rgb.mode,
    "local_env_file_present": ENV_FILE.is_file(),
}, indent=2))


## 3. Command and report helpers

Commands run through `uv run --no-sync` with the repository as their working directory. In uv mode, when `.env` exists, uv loads it for the child process; managed mode inherits the kernel environment; its contents are never read or printed by these helpers. Arguments are passed directly without a shell.

Output streams into the cell. A nonzero child exit raises `CalledProcessError`. Interruption forwards a stop signal to the subprocess group on POSIX. For a resumable planned stop, configure `runtime.wall_time_seconds`; interrupting a cell may leave only an earlier saved checkpoint.


In [ ]:
def stop_process(process):
    """Stop this command and its process group on POSIX when a cell is interrupted."""
    if process.poll() is not None:
        return
    try:
        if os.name == "posix":
            os.killpg(process.pid, signal.SIGINT)
        else:
            process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            if os.name == "posix":
                os.killpg(process.pid, signal.SIGKILL)
            else:
                process.kill()
            process.wait()
    except ProcessLookupError:
        process.wait()


def run_cli(stage, *arguments, execute=False, allow_network=False):
    """Preview by default; execute only this explicitly enabled CLI invocation."""
    if allow_network and stage != "prepare-model":
        raise ValueError("allow_network is only supported for prepare-model.")
    if "--allow-network" in map(str, arguments):
        raise ValueError("Use allow_network=True for the explicit model-download opt-in.")

    if RUNNER == "uv":
        argv = ["uv", "run", "--no-sync"]
        if ENV_FILE.is_file():
            argv += ["--env-file", str(ENV_FILE)]
        argv += ["python", "-m", "isl_slt"]
    else:
        argv = [sys.executable, "-m", "isl_slt"]
    argv += [stage, "--config", str(CONFIG_FILE)]
    argv += [str(argument) for argument in arguments]
    if stage in ("train", "overfit") and TRAINING_SEED is not None:
        argv += ["--seed", str(TRAINING_SEED)]
    if allow_network:
        argv += ["--allow-network"]

    print(shlex.join(argv), flush=True)
    if not execute:
        print("Preview only. Set execute=True in this stage cell when ready.")
        if stage == "prepare-model" and not allow_network:
            print("Model preparation also requires allow_network=True.")
        return None
    if stage == "prepare-model" and not allow_network:
        raise ValueError("Downloads require execute=True and allow_network=True.")
    if RUNNER == "uv" and shutil.which("uv") is None:
        raise FileNotFoundError("uv is not on PATH; finish terminal setup first.")

    environment = {**os.environ, "PYTHONUNBUFFERED": "1"}
    with subprocess.Popen(
        argv, cwd=REPO_ROOT, env=environment,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1,
        start_new_session=(os.name == "posix"),
    ) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
            return_code = process.wait()
        except BaseException:
            stop_process(process)
            raise
    if return_code:
        raise subprocess.CalledProcessError(return_code, argv)
    return return_code


def show_json(path, max_characters=12000):
    """Display an existing local report; never create or fabricate one."""
    path = repo_path(path)
    if not path.is_file():
        print(f"No saved report yet: {path}")
        return
    with path.open(encoding="utf-8") as stream:
        report = json.load(stream)
    rendered = json.dumps(report, indent=2, ensure_ascii=False)
    print(f"Saved report: {path}")
    print(rendered[:max_characters])
    if len(rendered) > max_characters:
        print("Display truncated; inspect the saved file for the full report.")


def preview_jsonl(path, limit=5):
    """Read at most limit nonempty records from an existing JSONL file."""
    path = repo_path(path)
    if not path.is_file():
        print(f"No saved records yet: {path}")
        return
    print(f"Saved records: {path}")
    shown = 0
    with path.open(encoding="utf-8") as stream:
        for line in stream:
            if not line.strip():
                continue
            print(json.dumps(json.loads(line), indent=2, ensure_ascii=False))
            shown += 1
            if shown >= limit:
                break
    if not shown:
        print("The saved file is empty.")


## 4. Environment and configuration checks

Run the environment probe on the actual host before considering mixed precision. It executes recurrent forward/backward checks when enabled. The plan command resolves configuration but does not validate assets or numerical behavior. Initial baseline precision remains FP32.


In [ ]:
run_cli("env-check", execute=False)


In [ ]:
run_cli("plan", execute=False)


### Source candidates before audit

This exports a separate unverified candidate map and provenance report. Review real naming conventions and cross-dataset evidence, then supply an authoritative mapping. The command never marks regex matches verified automatically. Use --dataset pretrain with the transfer config for its extra annotations.


In [ ]:
run_cli("make-source-map", execute=False)


## 5. Audit the supplied data

Audit uses the selected configuration, including transfer adapters or context/RGB metadata when configured. It creates immutable manifests and audit reports. Review missing poses, invalid labels, source provenance, interpreter disagreements and requested timing/geometry availability before splitting.

Existing report files may come from an earlier preparation; inspect their revisions and hashes. A report containing issues is not a successful validation.


In [ ]:
run_cli("audit", execute=False)


In [ ]:
show_json(repo_path(CONFIG.data.manifest).with_suffix(".audit.json"))
if CONFIG.data.pretrain is not None:
    show_json(repo_path(CONFIG.data.pretrain_manifest).with_suffix(".audit.json"))


## 6. Freeze memberships and check leakage

Execute splitting and leakage checking separately, then inspect both reports. Train seeds reuse these immutable memberships. Resolve provenance/leakage issues rather than editing frozen files or silently reshuffling clips.


In [ ]:
run_cli("split", execute=False)


In [ ]:
run_cli("check-leakage", execute=False)


In [ ]:
show_json(SPLIT_DIR / "splits.json")
show_json(SPLIT_DIR / "leakage.json")
if CONFIG.data.pretrain is not None:
    show_json(SPLIT_DIR / "pretraining-filter.json")


## 7. Prepare pose features

This writes fingerprinted full-sequence feature caches for the selected feature/context schema. Supply real metadata before freezing manifests. Missing or stale assets must be resolved from their provenance. Input acquisition and extraction remain external preparation tasks.


In [ ]:
run_cli("prepare-features", execute=False)


### Cache integrity and optional statistics

Ordinary cached training/evaluation checks manifest-bound caches without raw files. verify-cache below explicitly verifies raw assets; add --no-verify-assets for a cache-only check. It does not disable dataset integrity checks. For standardize=true, prepare-stats fits continuous valid channels on target training only and embeds their identity in later checkpoints. Pose cache v2 requires regeneration of older caches.


In [ ]:
run_cli("verify-cache", execute=False)


In [ ]:
if CONFIG.features.standardize:
    run_cli("prepare-stats", execute=False)
else:
    print("Continuous-channel standardization is not enabled.")


## 8. Explicit model preparation

This is the only runner stage that can download model weights. Enable **both** `execute=True` and `allow_network=True` when future downloads are authorized. T5 preparation records the actual resolved commit and file checksums; RGB also prepares the declared ResNet18 initializer.

The attention-LSTM experiment needs no T5 initializer. No weights or resource downloads run automatically.


In [ ]:
if CONFIG.model.decoder == "t5" or CONFIG.rgb.mode != "off":
    run_cli("prepare-model", execute=False, allow_network=False)
else:
    print("No pretrained model download is required for this configuration.")


In [ ]:
if CONFIG.model.decoder == "t5":
    show_json(repo_path(CONFIG.model.t5_dir) / "asset.json")


## 9. Optional frozen RGB extraction

Only cached RGB mode needs this command. Image mode reads actual synchronized crops during training instead. Both modes require real videos and verified pose-to-video transforms in frozen manifests. Install the `rgb` extra during terminal setup when using either RGB mode.


In [ ]:
if CONFIG.rgb.mode == "cached":
    run_cli("prepare-rgb", execute=False)
else:
    print("Frozen RGB extraction is not required for this configuration.")


## 10. Training

Inspect audit/leakage reports and prepared assets first. The dry-run below prints a resolved plan; it does not test a model, load a dataset or establish numerical correctness. Follow the runbook's separate [small-subset check](../docs/runbook.md#6-inspect-a-small-training-subset) before a full run.

Use fresh training **or** the resume cell, according to the trajectory's state. Every cell remains a preview until individually enabled. The CLI rejects a new run in an existing run directory.

Per-epoch validation is greedy by default. The explicit validation cell below performs the final full beam sweep. Overfit, velocity removal and train-only standardization are separate explicit configs; none silently changes the baseline.


In [ ]:
run_cli("train", "--dry-run", execute=False)


In [ ]:
run_cli("train", execute=False)


## 11. Resume a matching trajectory

This restores `last.pt` for the selected config and optional seed directory. Resume requires the matching backend, numerical policy, data/cache/tokenizer identities and experiment configuration. Averaged checkpoints cannot resume training.

Code/source and actual environment/lock identity are checked. `ALLOW_CODE_CHANGE=True` records only a code transition; numerical/data/environment mismatches still fail.


In [ ]:
resume_arguments = ["--resume", RESUME_CHECKPOINT]
if ALLOW_CODE_CHANGE:
    resume_arguments += ["--allow-code-change"]
run_cli("train", *resume_arguments, execute=False)


## 12. Free-generation evaluation

Select `EVALUATION_PARTITION` in the configuration cell. Per-epoch decoding is cheaper development selection. Run the explicit final validation sweep first; test/gold reuse its checkpoint-bound decoding.json. Scores use the chosen config's metrics. Missing requested resources are reported explicitly, and the CLI saves predictions/reports before returning an error for unavailable metrics.

Baseline METEOR needs local WordNet. Prepare it explicitly later from the terminal if not already available:

```bash
uv run --no-sync --env-file .env python -m nltk.downloader -d artifacts/nltk_data wordnet
```

The following optional cell performs fresh **validation-only** decoding selection for the chosen checkpoint, including averaged weights. For later test/gold evaluation with that selection, set `DECODING_FILE` to the displayed `validation_decoding_file` path and rerun configuration/helpers. Test/gold references never select decoding.

Gold scoring includes transcript results for the full holdout and paired transcript/interpreter results for the same annotated UIDs. Primary cased metrics and `_lc` variants carry separate signatures. No fixed 397/291 counts are assumed.


In [ ]:
run_cli(
    "evaluate", "--checkpoint", CHECKPOINT,
    "--partition", "validation", "--output", VALIDATION_OUTPUT,
    execute=False,
)


In [ ]:
evaluation_arguments = [
    "--checkpoint", CHECKPOINT,
    "--partition", EVALUATION_PARTITION,
    "--output", EVALUATION_OUTPUT,
]
if EVALUATION_PARTITION != "validation":
    selection_path = repo_path(DECODING_FILE) if DECODING_FILE is not None else VALIDATION_OUTPUT / "decoding.json"
    evaluation_arguments += ["--decoding", selection_path]
run_cli("evaluate", *evaluation_arguments, execute=False)


## 13. Optional paired visual diagnostics

This uses the selected experiment's model/feature configuration and validation-selected trajectory checkpoint. It compares real, blank, incorrect and shuffled visual inputs, recording actual paired predictions. It is evidence of input dependence, not proof of semantic faithfulness. Averaged checkpoints use the same explicit validation-selection file through --decoding.


In [ ]:
selection_path = repo_path(DECODING_FILE) if DECODING_FILE is not None else VALIDATION_OUTPUT / "decoding.json"
run_cli(
    "diagnostics", "--checkpoint", CHECKPOINT,
    "--partition", EVALUATION_PARTITION, "--output", DIAGNOSTIC_OUTPUT,
    "--decoding", selection_path, execute=False,
)


## 14. Inspect actual saved outputs

These cells read only existing local reports and a bounded preview of predictions/events. Missing files are shown as absent. Evaluation output is separated by checkpoint stem and partition under the selected run directory. Gold transcript/interpreter scores remain separate inside the report.

Inspect metric availability, signatures, split hashes and raw predictions before interpreting any score. Retain the real lockfile, resolved configuration, asset provenance, checkpoints and evaluation files together. Clear notebook outputs and execution counts before committing or sharing the notebook.


In [ ]:
show_json(RUN_DIR / "resolved.json")
show_json(RUN_DIR / "compatibility.json")
show_json(EVALUATION_OUTPUT / "metrics.json")
preview_jsonl(EVALUATION_OUTPUT / "predictions.jsonl")
preview_jsonl(RUN_DIR / "events.jsonl")
show_json(DIAGNOSTIC_OUTPUT / "diagnostics.json")


## Future verification

No notebook execution or numerical validation has occurred. Later verification should cover:

- Opening from the repository root and from `notebooks/`, with the correct uv-managed kernel.
- Baseline and local-hand selection, including derived run/checkpoint/output paths and optional seed overrides.
- Preview cells starting no subprocesses, and model downloads requiring the additional network opt-in.
- Missing configuration, inputs, weights or scoring resources producing clear failures without fake results.
- Failed subprocesses raising errors and interrupted command groups stopping on the intended POSIX host.
- A matching saved training trajectory resuming correctly, followed by reference-isolated validation/test/gold evaluation.
- Actual saved-report inspection and cleared outputs/execution counts before sharing.

Offline CPU tests are restored under tests/ and remain unexecuted. Future setup uses uv sync --locked --extra cpu --extra dev, then uv run --no-sync pytest. There are no claimed scores, throughput measurements or BLEU guarantees.
